# 02 - Expert Demonstration Collection (M2)

**Group members:** TODO

Roll out the deterministic expert, save the dataset (with per-episode boundaries), and run EDA. Quality gate: >= 90% of episodes above two thirds of the eval mean.

In [ ]:
# Make src importable whether run from notebooks/ or project root
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
# On Colab: mount Drive and set PROJECT_DATA_ROOT before importing src
from src import config, seeding, envs, collect, eval, plotting
seeding.set_seed(0)
ENV_ID = 'Walker2d-v4'  # switch to 'Ant-v4' for the second environment
print('device', config.device(), '| env', ENV_ID)

In [ ]:
from stable_baselines3 import PPO
model = PPO.load(config.MODELS_DIR / f'ppo_expert_{ENV_ID}' / 'best_model')
out_dir = config.DATA_DIR / ENV_ID
data = collect.collect(model, ENV_ID, n_episodes=100, out_dir=out_dir, seed=0)
print('episodes', len(data['episode_returns']), '| transitions', len(data['observations']))
print('mean return %.1f' % data['episode_returns'].mean())

## Exploratory data analysis (spec Section 5.3)

In [ ]:
fig = plotting.dataset_eda(data['episode_returns'], data['actions'])
plotting.save(fig, config.OUTPUTS_DIR / f'dataset_analysis_{ENV_ID}.png')

## Quality gate

In [ ]:
import numpy as np
thr = (2/3) * data['episode_returns'].mean()
frac = np.mean(data['episode_returns'] > thr)
print('fraction above 2/3 mean: %.2f' % frac)
assert frac >= 0.9, 'dataset quality gate failed'